# درس ۱۱ - پروتکل عامل به عامل (A2A)


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## پروتکل A2A چیست؟

پروتکل **عامل به عامل (A2A)** یک استاندارد باز است که به عامل‌های هوش‌مصنوعی امکان می‌دهد ارتباط برقرار کنند، یکدیگر را کشف کنند و همکاری نمایند — حتی زمانی که روی چارچوب‌های مختلف ساخته شده‌اند یا توسط سرویس‌های متفاوت میزبانی می‌شوند.

مفاهیم کلیدی:

- **کشف** – عامل‌ها یک *Agent Card* منتشر می‌کنند که قابلیت‌هایشان را توصیف می‌کند، و باعث می‌شود برای سایر عامل‌ها (یا هماهنگ‌کننده‌ها) یافتن متخصص مناسب برای یک وظیفه آسان‌تر شود.
- **انتقال پیام** – عامل‌ها پیام‌های ساخت‌یافته را از طریق یک پروتکل مشترک رد و بدل می‌کنند، بنابراین یک درخواست از یک عامل می‌تواند توسط عامل دیگر فهمیده و انجام شود، بدون توجه به پیاده‌سازی داخلی آن.
- **چرخه عمر وظیفه** – A2A وضعیت‌هایی مانند *ارسال‌شده*, *درحال‌کار*, *تکمیل‌شده*, و *ناموفق* را تعریف می‌کند، و به هماهنگ‌کننده دید کامل می‌دهد از اینکه یک وظیفه واگذار شده چگونه پیش می‌رود.

در این درس ما همکاری به سبک A2A را با اتصال سه عامل تخصصی سفر به یک جریان کاری شبیه‌سازی می‌کنیم که در آن هر عامل تخصص خود را ارائه می‌دهد و نتایج را به عامل بعدی منتقل می‌کند.


## ایجاد نمایندگان مسافرتی تخصصی


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## همکاری چندعاملی از طریق جریان کاری

ما سه عامل را در یک جریان کاری متوالی که شبیه انتقال پیام A2A است، به هم متصل می‌کنیم:

1. **CurrencyExchangeAgent** درخواست کاربر را دریافت می‌کند و راهنمایی ارزی تولید می‌کند.
2. **ActivityPlannerAgent** زمینهٔ غنی‌شده را دریافت کرده و توصیه‌های فعالیت را اضافه می‌کند.
3. **TravelManagerAgent** هر دو ورودی را ترکیب کرده و یک خلاصه نهایی سفر تدوین می‌کند.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## درک A2A در محیط تولید

در یک محیط تولید پروتکل A2A سناریوهای قدرتمند میان‌خدمتی را فراهم می‌سازد:

| Capability | Description |
|---|---|
| **هم‌تعاملی بین‌چارچوبی** | عاملی که با یک چارچوب ساخته شده می‌تواند وظایف را به عاملی که با هر چارچوب سازگار با A2A ساخته شده واگذار کند، و امکان تعامل واقعی بین سازمان‌ها را فراهم آورد. |
| **مرزهای سرویس** | عامل‌ها می‌توانند در میکروسرویس‌های جداگانه، نواحی مختلف ابری، یا حتی سازمان‌های متفاوت مستقر باشند و در عین حال به‌صورت یکپارچه همکاری کنند. |
| **کشف پویا** | یک ارکستراتور می‌تواند در زمان اجرا از رجیستری Agent Card پرس‌وجو کند تا مناسب‌ترین متخصص را برای یک زیروظیفه مشخص بیابد. |
| **پخش جریانی و اعلان‌های پوش** | A2A از Server-Sent Events (SSE) برای به‌روزرسانی‌های پیشرفت در زمان واقعی و از اعلان‌های پوش برای وظایف طولانی‌مدت پشتیبانی می‌کند. |

جریان کاری که در بالا ساختیم نسخه‌ای ساده‌شده، درون‌پردازشی از این الگو است. در یک
استقرار واقعی هر عامل یک نقطه پایانی HTTP را افشا می‌کرد، یک Agent Card منتشر می‌نمود، و ارتباط
از طریق پروتکل A2A JSON-RPC برقرار می‌کرد.


## خلاصه

در این درس یاد گرفتید:

1. **A2A پروتکل چیست** — یک استاندارد باز برای کشف عامل‌به‌عامل، ارسال پیام،
   و مدیریت وظایف.
2. **چگونه عوامل تخصصی ایجاد کنیم** — یک عامل تبدیل ارز، یک عامل برنامه‌ریز فعالیت،
   و یک اورکستراتور مدیر سفر.
3. **چگونه عوامل را به یک جریان کاری متصل کنیم** — با استفاده از `WorkflowBuilder` برای مدل‌سازی انتقال پیام‌های متوالی
   بین عوامل.
4. **چگونه A2A در تولید عمل می‌کند** — فعال‌سازی همکاری میان‌چارچوبی و میان‌خدمتی
   با کشف پویا و به‌روزرسانی‌های جریان‌دار.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
سلب مسئولیت:
این سند با استفاده از سرویس ترجمهٔ هوش مصنوعی Co-op Translator (https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما برای دقت تلاش می‌کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطا یا نادرستی باشند. نسخهٔ اصلی سند به زبان مبدأ باید به‌عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حساس، ترجمهٔ حرفه‌ای و انسانی توصیه می‌شود. ما در قبال هرگونه سوتفاهم یا تفسیر نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
